In [2]:
! pip install pymupdf python-docx openpyxl


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


# DOCUMENTS LOAD

In [4]:
import fitz  # pymupdf


def load_pdf_as_text(path: str) -> str:
    """
    Загружает PDF и возвращает текст,
    разделённый по страницам.
    """

    doc = fitz.open(path)

    pages_text = []

    for page in doc:
        text = page.get_text("text")

        # минимальная нормализация
        text = text.replace("\r", "").strip()

        if text:
            pages_text.append(text)

    # разделяем страницы двойным переносом
    return "\n\n".join(pages_text)

In [5]:
from docx import Document


def load_docx_as_text(path: str) -> str:
    """
    Загружает DOCX и возвращает текст
    с сохранением структуры параграфов.
    """

    doc = Document(path)

    paragraphs = []

    for p in doc.paragraphs:
        text = p.text.strip()

        if not text:
            continue

        paragraphs.append(text)

    return "\n\n".join(paragraphs)

In [6]:
import os


def load_document(path: str) -> str:
    """
    Универсальный загрузчик PDF / DOCX.
    """

    ext = os.path.splitext(path)[1].lower()

    if ext == ".pdf":
        return load_pdf_as_text(path)

    if ext == ".docx":
        return load_docx_as_text(path)

    raise ValueError(f"Unsupported format: {ext}")

# DOCUMENTS PARSING

In [7]:
from dataclasses import dataclass
from typing import Optional


@dataclass
class Chunk:
    id: str
    doc_name: str
    section: str
    text: str
    level: str  # "chapter" | "article" | "point"
    parent_id: Optional[str] = None

In [8]:
import re

PATTERNS = {
    "chapter": re.compile(r"^(Глава\s+\d+\.?.*)$", re.MULTILINE),
    "section": re.compile(r"^(Раздел\s+[IVXLCDM\d]+\.?.*)$", re.MULTILINE),
    "article": re.compile(r"^(Статья\s+\d+(\.\d+)?\.?.*)$", re.MULTILINE),
    "point": re.compile(r"^(\d+(\.\d+)*\.)\s", re.MULTILINE),
}

In [9]:
def extract_structured_headers(text: str):

    headers = []

    for level, pattern in PATTERNS.items():

        for m in pattern.finditer(text):

            headers.append({
                "level": level,
                "title": m.group(1).strip(),
                "start": m.start()
            })

    return sorted(headers, key=lambda x: x["start"])

In [10]:
LEVEL_ORDER = ["section", "chapter", "article", "point"]


def _update_stack(stack, header, node_id, level):

    # убираем всё что глубже или равно текущему уровню
    while stack:
        last_level = stack[-1]["level"]

        if LEVEL_ORDER.index(last_level) >= LEVEL_ORDER.index(level):
            stack.pop()
        else:
            break

    stack.append({
        "id": node_id,
        "level": level
    })

In [11]:
import uuid

def build_chunk_graph(doc_name: str, text: str):

    headers = extract_structured_headers(text)

    if not headers:
        return [
            Chunk(
                id=str(uuid.uuid4()),
                doc_name=doc_name,
                section="DOCUMENT",
                text=text
            )
        ]

    chunks = []
    stack = []  # хранит текущую иерархию

    for i, h in enumerate(headers):

        start = h["start"]
        end = headers[i + 1]["start"] if i + 1 < len(headers) else len(text)

        content = text[start:end].strip()

        node_id = str(uuid.uuid4())

        # определяем parent
        parent_id = stack[-1]["id"] if stack else None

        chunk = Chunk(
            id=node_id,
            doc_name=doc_name,
            level=h["level"],
            section=h["title"],
            text=content,
            parent_id=parent_id
        )

        chunks.append(chunk)

        # обновляем стек по уровню
        _update_stack(stack, h, node_id, h["level"])

    return chunks

In [12]:
def expand_path(chunk, chunks_by_id):

    path = []
    seen_levels = set()

    current = chunk

    while current:

        # берём только 1 узел каждого уровня
        if current.level not in seen_levels:

            path.append(current)
            seen_levels.add(current.level)

        if not current.parent_id:
            break

        current = chunks_by_id.get(current.parent_id)

    return list(reversed(path))

In [13]:
test_path = "/Users/z123/Desktop/rag-agent/personal_data.pdf"

In [14]:
text = load_document(test_path)

chunks = build_chunk_graph(doc_name=test_path.split('/')[-1], text = text)

In [15]:
chunks_by_id = {chunk.id: chunk for chunk in chunks}

In [16]:
expand_path(chunk=chunks[10], chunks_by_id=chunks_by_id)

[Chunk(id='54b503ca-3ec2-4f29-b712-83505bf1f2c1', doc_name='personal_data.pdf', section='Глава 1. ОБЩИЕ ПОЛОЖЕНИЯ', text='Глава 1. ОБЩИЕ ПОЛОЖЕНИЯ', level='chapter', parent_id=None),
 Chunk(id='01ec613d-3176-42c8-9e9a-4e3a24c43c1e', doc_name='personal_data.pdf', section='Статья 4. Законодательство Российской Федерации в области', text='Статья 4. Законодательство Российской Федерации в области \nперсональных данных', level='article', parent_id='1db72af1-7948-41cc-86b8-5e3ba57d25b0'),
 Chunk(id='a1ebe531-2bc7-4d96-980e-633d7795ee98', doc_name='personal_data.pdf', section='3.', text='3. Особенности обработки персональных данных, осуществляемой \nбез использования средств автоматизации, могут быть установлены \nфедеральными законами и иными нормативными правовыми актами \nРоссийской Федерации с учетом положений настоящего Федерального \nзакона.', level='point', parent_id='c50fd2fa-2215-40c0-9c84-152e76d3ab8c')]

# SPARSE RETRIVAL

In [17]:
import re
import pymorphy3

morph = pymorphy3.MorphAnalyzer()

def normalize(text):

    words = re.findall(
        r"\w+",
        text.lower()
    )

    return [
        morph.parse(word)[0].normal_form
        for word in words
    ]

In [18]:
! pip install pymorphy3 rank-bm25


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [20]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [
    normalize(chunk.text)
    for chunk in chunks
]

bm25 = BM25Okapi(tokenized_chunks)

In [48]:
def sparce_search(query, k=5):

    tokens = normalize(query)

    scores = bm25.get_scores(tokens)

    ranked = sorted(
        enumerate(scores),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for idx, score in ranked[:k]:

        results.append({
            "score": float(score),
            "chunk": chunks[idx]
        })

    return results

In [49]:
test_query = 'уровни защищенности'

In [ ]:
expand_path(chunk=sparce_search(test_query)[0]['chunk'], chunks_by_id=chunks_by_id)

[Chunk(id='15fd6262-d279-49f9-984f-169bde170988', doc_name='personal_data.pdf', section='Глава 4. ОБЯЗАННОСТИ ОПЕРАТОРА', text='Глава 4. ОБЯЗАННОСТИ ОПЕРАТОРА', level='chapter', parent_id='2430ac69-229b-434d-8128-1de890298ea9'),
 Chunk(id='f88fe544-52bf-4c30-bf05-b8fbf23d5a31', doc_name='personal_data.pdf', section='Статья 19. Меры по обеспечению безопасности персональных', text='Статья 19. Меры по обеспечению безопасности персональных \nданных при их обработке \n \n(в ред. Федерального закона от 25.07.2011 N 261-ФЗ)', level='article', parent_id='a03fb166-033a-4d66-89e2-40d8af3a418a'),
 Chunk(id='0bccf42f-8e2d-40c9-bca0-a433f496896b', doc_name='personal_data.pdf', section='3.', text='3. Правительство Российской Федерации с учетом возможного \nвреда \nсубъекту \nперсональных \nданных, \nобъема \nи \nсодержания \nобрабатываемых персональных данных, вида деятельности, при \nосуществлении \nкоторого \nобрабатываются \nперсональные \nданные, \nактуальности угроз безопасности персональных да

In [51]:
sparce_ids = [result['chunk'].id for result in sparce_search(test_query)]

# DENSE RETRIVAL

In [ ]:
! pip install sentence-transformers

In [53]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("intfloat/multilingual-e5-small")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5495.42it/s]


In [54]:
def embed_query(text: str):
    return model.encode(
        f"query: {text}",
        normalize_embeddings=True
    )


def embed_doc(text: str):
    return model.encode(
        f"passage: {text}",
        normalize_embeddings=True
    )

In [55]:
def build_dense_index(chunks):

    texts = [
        f"passage: {c.section}\n{c.text}"
        for c in chunks
    ]

    embeddings = model.encode(
        texts,
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=True
    )

    return np.array(embeddings)

In [56]:
def dense_search(query, k=5):

    chunk_embeddings = build_dense_index(chunks)

    q = embed_query(query)

    scores = np.dot(chunk_embeddings, q)

    top_idx = np.argsort(scores)[::-1][:k]

    return [
        {
            "score": float(scores[i]),
            "chunk": chunks[i]
        }
        for i in top_idx
    ]

In [58]:
dense_ids = [result['chunk'].id for result in dense_search(test_query)]

Batches: 100%|██████████| 5/5 [00:01<00:00,  4.93it/s]


# RECIPROCAL RANK FUSION

In [59]:
def rrf(rank, k=60):
    return 1.0 / (k + rank)

In [60]:
scores = {}

for rank, chunk_id in enumerate(sparce_ids, start=1):

    scores[chunk_id] = (
        scores.get(chunk_id, 0)
        + rrf(rank)
    )

for rank, chunk_id in enumerate(dense_ids, start=1):

    scores[chunk_id] = (
        scores.get(chunk_id, 0)
        + rrf(rank)
    )

In [61]:
ranked = sorted(
    scores.items(),
    key=lambda x: x[1],
    reverse=True
)
ranked

[('0bccf42f-8e2d-40c9-bca0-a433f496896b', 0.03252247488101534),
 ('d8207526-b799-478b-b83a-08839808bcdb', 0.032018442622950824),
 ('42e4e668-072f-4168-a7f5-0d54977290e7', 0.031754032258064516),
 ('7c3be7cf-5ff7-4db4-a68b-48e870220c0e', 0.031746031746031744),
 ('54b503ca-3ec2-4f29-b712-83505bf1f2c1', 0.015384615384615385),
 ('7ab396a2-784e-4af6-909c-decde90bf706', 0.015384615384615385)]